# YOLO-TRP Assignment (Google Colab)

Run cells in order. Before running, switch to **Runtime -> Change runtime type -> GPU**.

In [ ]:
!nvidia-smi
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
print('Device count:', torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime not enabled. Switch runtime to GPU and reconnect.')

## Clone Your Repository

Set `REPO_URL` and (optionally) `BRANCH` below.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/<your-username>/<your-repo>.git'  # <- change this
BRANCH = 'main'
ASSIGNMENT_DIR = '/content/repo/_University/Semester-VI/DL_2026/Assignment-1'

# Always switch to stable cwd before deleting clone.
os.chdir('/content')
if os.path.exists('/content/repo'):
    subprocess.run(['rm', '-rf', '/content/repo'], check=True)

subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, '/content/repo'], check=True)
if not os.path.isdir(ASSIGNMENT_DIR):
    raise FileNotFoundError(f'Assignment folder not found: {ASSIGNMENT_DIR}')
os.chdir(ASSIGNMENT_DIR)
print('Working directory:', os.getcwd())

## Install Dependencies

This mirrors your local pipeline: install project deps, then force CUDA PyTorch wheels.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA still unavailable after install.')

## Drive Mount + Persistent Checkpoints

Mount Drive and set persistent directories for checkpoints/results.

All training checkpoints will be stored on Drive and survive Colab disconnects.`

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Update dataset root to your Drive path layout (train/images, val/images, test/images).
DATA_ROOT = '/content/drive/MyDrive/Colab Notebooks/DL_Dataset'

# Persist all model runs/checkpoints/results on Drive.
PERSIST_ROOT = '/content/drive/MyDrive/Colab Notebooks/DL_Outputs'
RUNS_DIR = os.path.join(PERSIST_ROOT, 'runs', 'detect')
RESULTS_DIR = os.path.join(PERSIST_ROOT, 'results')
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Export env vars consumed by project scripts.
os.environ['YOLO_RUNS_DIR'] = RUNS_DIR
os.environ['YOLO_RESULTS_DIR'] = RESULTS_DIR

print('YOLO_RUNS_DIR   =', os.environ['YOLO_RUNS_DIR'])
print('YOLO_RESULTS_DIR=', os.environ['YOLO_RESULTS_DIR'])

# Patch dataset.yaml for Colab Drive paths.
import yaml
YAML_PATH = os.path.join('/content/repo/_University/Semester-VI/DL_2026/Assignment-1', 'dataset.yaml')
with open(YAML_PATH, 'r') as f:
    d = yaml.safe_load(f)

d['path'] = DATA_ROOT
d['train'] = 'train/images'
d['val'] = 'val/images'
d['test'] = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(d, f, sort_keys=False)

print('\nUpdated dataset.yaml:\n')
print(open(YAML_PATH, 'r').read())

## Run Full Pipeline

In [ ]:
!chmod +x run_pipeline.ps1 || true

# PowerShell script is Windows-specific, so run Python steps directly on Colab Linux.
!python 00_sanitize_labels.py
!python 01_explore_dataset.py
!python 02_train_baseline.py
!python 03_train_custom.py
!python 04_evaluate.py
!python 05_compare_results.py

## Package Outputs

In [ ]:
# Package from persistent Drive outputs + key project files.
!zip -r assignment_outputs.zip "$YOLO_RESULTS_DIR" "$YOLO_RUNS_DIR" report *.py custom_model dataset.yaml requirements.txt
print('Created assignment_outputs.zip')

## Download Outputs

In [ ]:
from google.colab import files
files.download('assignment_outputs.zip')